# Tensor-network search for two-dimensional square-QDM cage states

This notebook is intentionally separate from `cage_padding.ipynb`.  It starts from the exact constrained vertex-PEPS manifold, embeds the two-plaquette singlet as a sparse initialization, differentiates the exact finite-cluster Hamiltonian variance with Autograd, and uses quimb to optimize the shared unit tensor.

The present optimization is a **discovery calculation**, not yet a proof of a thermodynamic eigenstate.  A candidate must later pass larger-cluster transfer tests and a local telescoping/eigenstate certificate.

## Installation

Install the optional tensor-network stack with

```bash
pip install "qlinks[tn]"
```

The `tn` extra includes `quimb`, `autograd`, `numba`, and `llvmlite`.  It currently targets Python 3.11--3.13 so that Intel macOS can use the last available binary `llvmlite` wheels rather than compiling LLVM.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

import autograd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import quimb

from qlinks.caging import (
    SquareQDMChiralPEPSAnsatz,
    SquareQDMPEPSAnsatz,
    autograd_available,
    build_square_qdm_peps_finite_cluster_problem,
    build_square_qdm_rectangular_tile_tensor_basis,
    build_square_qdm_singlet_peps_ansatz,
    build_square_qdm_type1_peps_problem,
    quimb_available,
    square_qdm_two_plaquette_singlet_blocks,
)
from qlinks.models import SquareQDMModel
from qlinks.visualizer import SquareQDMTensorNetworkVisualizer

print("quimb:", quimb.__version__)
print("autograd available:", autograd_available())
print("quimb available:", quimb_available())

## 1. Exact constrained unit tensor

A $3\times2$ tile owns the outgoing $+x$ and $+y$ links of its six vertices.  The virtual indices carry the boundary dimer occupations.  The structural mask therefore enforces the dimer constraint exactly when the tile is repeated in both directions.

In [ ]:
host = SquareQDMModel(
    lx=8,
    ly=8,
    boundary_condition="periodic",
    coup_kin=1.0,
    coup_pot=0.0,
)

tile_basis = build_square_qdm_rectangular_tile_tensor_basis(
    host,
    tile_shape=(3, 2),
    origin=(2, 2),
)

pd.Series({
    "tile shape": tile_basis.tile_shape,
    "owned links": tile_basis.owned_link_ids.size,
    "unconstrained configurations": 2 ** tile_basis.owned_link_ids.size,
    "physical states": tile_basis.physical_dimension,
    "allowed tensor entries": tile_basis.n_entries,
    "tensor shape (u,r,d,l,p)": tile_basis.tensor_shape,
})

In [ ]:
visualizer = SquareQDMTensorNetworkVisualizer(tile_basis)
visualizer.plot_network(n_tiles_x=3, n_tiles_y=2)
plt.show()

## 2. Structural PEPS benchmark

Setting every allowed tensor entry to one produces the equal-weight superposition of all valid dimer coverings.  On a $2\times2$ tile array, corresponding to a periodic $6\times4$ lattice, the network norm must equal the exact constrained-basis dimension.

In [ ]:
structural_ansatz = SquareQDMPEPSAnsatz(
    tile_basis=tile_basis,
    parameters=np.ones(tile_basis.n_entries, dtype=np.complex128),
)
structural_network = structural_ansatz.to_quimb_tensor_network(
    n_tiles_x=2,
    n_tiles_y=2,
)
structural_norm = float(np.real(structural_network.norm(squared=True, optimize="greedy")))

full_6x4 = SquareQDMModel(
    lx=6,
    ly=4,
    boundary_condition="periodic",
    coup_kin=1.0,
    coup_pot=0.0,
)

pd.Series({
    "PEPS norm squared": structural_norm,
    "exact dimer coverings": full_6x4.build_basis().n_states,
    "agreement": np.isclose(structural_norm, full_6x4.build_basis().n_states),
})

## 3. Embed the local two-plaquette singlet

The bare singlet occupies only two of the 108 allowed tensor entries.  Repeating it in two dimensions gives a valid dimer state, but bridge plaquettes produce a nonzero Hamiltonian variance.

In [ ]:
singlet = next(
    block
    for block in square_qdm_two_plaquette_singlet_blocks(host, directions=("x",))
    if set(block.anchor_cells) == {(2, 2), (3, 2)}
)

singlet_ansatz = build_square_qdm_singlet_peps_ansatz(
    host,
    singlet,
    origin=(2, 2),
)

nonzero_entries = np.flatnonzero(np.abs(singlet_ansatz.parameters) > 1.0e-12)
print("nonzero singlet entries:", nonzero_entries.tolist())

fig, axes = plt.subplots(1, len(nonzero_entries), figsize=(10, 4))
for axis, entry_index in zip(np.atleast_1d(axes), nonzero_entries, strict=True):
    visualizer.plot_entry(int(entry_index), ax=axis)
plt.tight_layout()
plt.show()

In [ ]:
visualizer.plot_parameter_magnitudes(
    singlet_ansatz.parameters,
    max_entries=16,
    title="Sparse two-plaquette singlet initialization",
)
plt.show()

## 4. Exact $6\times4$ optimization problem

The loss is the exact normalized energy variance

\[
\mathcal V_H(A)=
\frac{\langle\Psi(A)|(H-\langle H\rangle_A)^2|\Psi(A)\rangle}
{\langle\Psi(A)|\Psi(A)\rangle}.
\]

The finite cluster is evaluated in the $w_{00}$ sector.  The compact quimb `PTensor` exposes only the 108 structurally allowed parameters.

In [ ]:
model_6x4_w00 = SquareQDMModel(
    lx=6,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=1.0,
    coup_pot=0.0,
)

problem = build_square_qdm_peps_finite_cluster_problem(
    model_6x4_w00,
    tile_basis,
)

singlet_report = problem.diagnose(singlet_ansatz.parameters)
pd.Series({
    "Hilbert dimension": problem.hilbert_dimension,
    "nonzero amplitudes": singlet_report.nonzero_basis_amplitudes,
    "energy": singlet_report.energy.real,
    "variance": singlet_report.energy_variance,
    "residual": singlet_report.residual,
})

### Why perturb the singlet?

The exactly sparse singlet tensor is stationary inside the enlarged manifold: new entries first contribute through products around the periodic tile array.  A small reproducible perturbation activates the boundary-compatible sectors and gives Autograd a nonzero gradient.

In [ ]:
initial_parameters = problem.perturb_parameters(
    singlet_ansatz.parameters,
    scale=1.0e-2,
    seed=0,
)
initial_loss, initial_gradient = problem.loss_and_gradient_autograd(initial_parameters)
optimizer = problem.make_quimb_optimizer(initial_parameters, progbar=False)

pd.Series({
    "compact optimizer dimension": optimizer.d,
    "initial perturbed variance": initial_loss,
    "gradient norm": np.linalg.norm(initial_gradient),
})

## 5. Short quimb/Autograd demonstration

Five L-BFGS-B iterations are short enough for interactive use.  Increase `OPTIMIZATION_STEPS` for an actual search.  A decreasing loss only shows that the larger tensor manifold repairs part of the bridge leakage; zero variance and cross-size stability are required before interpreting a solution as a cage state.

In [ ]:
RUN_OPTIMIZATION = False
OPTIMIZATION_STEPS = 5

if RUN_OPTIMIZATION:
    optimization = problem.optimize_with_quimb(
        singlet_ansatz.parameters,
        max_steps=OPTIMIZATION_STEPS,
        noise_scale=1.0e-2,
        seed=0,
        autodiff_backend="autograd",
        optimizer="L-BFGS-B",
        progbar=False,
    )
    display(pd.Series({
        "initial variance": optimization.initial_loss,
        "final variance": optimization.final_loss,
        "initial residual": optimization.initial_report.residual,
        "final residual": optimization.final_report.residual,
        "function evaluations": len(optimization.loss_history),
        "improved": optimization.improved,
        "exact within tolerance": optimization.reached_exact_state,
    }))
else:
    optimization = None
    print("Set RUN_OPTIMIZATION=True to run the demonstration.")

In [ ]:
if optimization is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    visualizer.plot_optimization_history(optimization, ax=axes[0], log_scale=False)
    visualizer.plot_parameter_magnitudes(
        optimization.optimized_parameters,
        max_entries=24,
        ax=axes[1],
        title="Largest optimized unit-tensor amplitudes",
    )
    plt.tight_layout()
    plt.show()

## 6. Candidate-validation workflow

For any low-variance tensor found here:

1. rerun multiple random seeds and remove gauge/normalization redundancies;
2. validate the same unit tensor on $9\times4$, $6\times6$, and $9\times6$ clusters;
3. inspect whether the optimized support retains the cage-derived local annihilator $L_R$;
4. reconstruct simple exact coefficients where possible;
5. derive a local PEPS telescoping identity proving the eigenstate equation for arbitrary $n_x,n_y$;
6. only then combine the state certificate with the nonzero thermal $\langle Q_R\rangle$ calculation.

The visualization API is intended to make steps 1--3 inspectable: it draws the repeated tensor graph, individual boundary-resolved dimer entries, compact tensor amplitudes, and the optimization trace.

## 7. Type-1 cage structure in the PEPS objective

A type-1 cage is not searched as a generic low-variance state.  We project the PEPS amplitudes onto one kinetic-graph chiral subset and evaluate the two defining conditions separately:

\[
\mathcal L_K=\frac{\|K|\Psi_+\rangle\|^2}{N_p\langle\Psi_+|\Psi_+\rangle},
\qquad
\mathcal L_V=\frac{\operatorname{Var}_{\Psi_+}(V)}{N_p}.
\]

The first term is the destructive-interference residual on the empty chiral subset.  The second tests whether the diagonal potential is uniform on the occupied support.  The chiral projection itself is exact on the finite constrained basis.  In addition, qlinks infers a tile-periodic link-parity rule so the same symmetry can later be encoded natively in the PEPS tensor.

In [ ]:
type1_problem = build_square_qdm_type1_peps_problem(
    model_6x4_w00,
    tile_basis,
    reference_parameters=singlet_ansatz.parameters,
)

type1_report = type1_problem.diagnose(singlet_ansatz.parameters)
local_chiral_charges = type1_problem.parity_rule.tile_physical_charges(
    model_6x4_w00,
    tile_basis,
)

pd.Series({
    "selected chiral subset": type1_problem.target_chiral_label,
    "retained chiral weight": type1_report.retained_chiral_weight,
    "discarded chiral weight": type1_report.discarded_chiral_weight,
    "kinetic-interference norm": type1_report.kinetic_interference_norm,
    "kinetic-interference density": type1_report.kinetic_interference_density,
    "potential variance density": type1_report.potential_variance_density,
    "type-1 objective density": type1_report.objective,
    "nonzero opposite-sector residuals": type1_report.n_nonzero_interference_targets,
    "tile-periodic chiral rule": type1_problem.parity_rule.metadata.get("tile_periodic"),
    "C=+ local physical states": int(np.count_nonzero(local_chiral_charges == 0)),
    "C=- local physical states": int(np.count_nonzero(local_chiral_charges == 1)),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
visualizer.plot_type1_components(type1_report, ax=axes[0])
visualizer.plot_chiral_physical_charges(
    type1_problem.parity_rule,
    model_6x4_w00,
    ax=axes[1],
)
plt.tight_layout()
plt.show()

### Potential-uniformity diagnostic

For the pure kinetic model, \(V=0\) and the second condition is automatic.  Turning on the plaquette potential exposes the other type-1 requirement without changing the chiral projection.

In [ ]:
model_6x4_v1 = SquareQDMModel(
    lx=6,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=1.0,
    coup_pot=1.0,
)

type1_problem_v1 = build_square_qdm_type1_peps_problem(
    model_6x4_v1,
    tile_basis,
    reference_parameters=singlet_ansatz.parameters,
)
type1_report_v1 = type1_problem_v1.diagnose(singlet_ansatz.parameters)

pd.DataFrame(
    {
        "pure kinetic": {
            "kinetic density": type1_report.kinetic_interference_density,
            "potential mean": type1_report.potential_mean,
            "potential variance density": type1_report.potential_variance_density,
            "objective": type1_report.objective,
        },
        "K + V": {
            "kinetic density": type1_report_v1.kinetic_interference_density,
            "potential mean": type1_report_v1.potential_mean,
            "potential variance density": type1_report_v1.potential_variance_density,
            "objective": type1_report_v1.objective,
        },
    }
)

### Type-1-specific optimization

The optimizer below no longer minimizes a generic energy variance.  It varies the 108 allowed tensor entries while the finite-cluster state is kept in one chiral sector, and minimizes kinetic interference plus potential nonuniformity.  Keep the run disabled until the diagnostic cells above look sensible on your machine.

In [ ]:
type1_initial = type1_problem.base_problem.perturb_parameters(
    singlet_ansatz.parameters,
    scale=1.0e-2,
    seed=0,
)
type1_loss, type1_gradient = type1_problem.loss_and_gradient_autograd(type1_initial)

pd.Series({
    "initial type-1 objective density": type1_loss,
    "gradient norm": np.linalg.norm(type1_gradient),
})

In [ ]:
RUN_TYPE1_OPTIMIZATION = False
TYPE1_OPTIMIZATION_STEPS = 10

if RUN_TYPE1_OPTIMIZATION:
    type1_optimization = type1_problem.optimize_with_quimb(
        singlet_ansatz.parameters,
        max_steps=TYPE1_OPTIMIZATION_STEPS,
        noise_scale=1.0e-2,
        seed=0,
        autodiff_backend="autograd",
        optimizer="L-BFGS-B",
        progbar=False,
    )
    display(pd.DataFrame({
        "initial": {
            "kinetic density": type1_optimization.initial_report.kinetic_interference_density,
            "potential variance density": type1_optimization.initial_report.potential_variance_density,
            "objective": type1_optimization.initial_report.objective,
        },
        "final": {
            "kinetic density": type1_optimization.final_report.kinetic_interference_density,
            "potential variance density": type1_optimization.final_report.potential_variance_density,
            "objective": type1_optimization.final_report.objective,
        },
    }))
    visualizer.plot_type1_optimization_history(type1_optimization, log_scale=False)
    plt.show()
else:
    type1_optimization = None
    print("Set RUN_TYPE1_OPTIMIZATION=True to run the type-1-specific search.")

### Native \(\mathbb Z_2\)-symmetric tensor

The finite-cluster projector is useful for optimization, but the inferred chiral parity is also compatible with the \(3\times2\) tile itself.  qlinks therefore augments each virtual leg by one \(\mathbb Z_2\) charge bit and keeps only tensor entries satisfying local charge conservation.  Contracting a closed torus cancels all virtual charges pairwise, leaving support only in the selected global chiral subset.

This does not add variational parameters: each of the 108 structural amplitudes is copied into eight parity-compatible charge sectors.

In [ ]:
native_chiral_ansatz = SquareQDMChiralPEPSAnsatz.from_type1_problem(
    type1_problem,
    singlet_ansatz.parameters,
)

pd.Series({
    "base variational parameters": native_chiral_ansatz.n_parameters,
    "charge-augmented tensor shape": native_chiral_ansatz.tensor_shape,
    "nonzero charge-resolved entries": native_chiral_ansatz.n_nonzero_tensor_entries,
    "virtual-charge degeneracy on 2x2 torus": native_chiral_ansatz.charge_degeneracy(
        n_tiles_x=2,
        n_tiles_y=2,
    ),
})

In [ ]:
RUN_NATIVE_CHIRAL_CONTRACTION = False

if RUN_NATIVE_CHIRAL_CONTRACTION:
    native_network = native_chiral_ansatz.to_quimb_tensor_network(
        n_tiles_x=2,
        n_tiles_y=2,
    )
    native_norm_squared = native_network.norm(squared=True, optimize="greedy")
    print("native chiral PEPS norm squared:", native_norm_squared)
else:
    print(
        "Set RUN_NATIVE_CHIRAL_CONTRACTION=True to contract the charge-augmented "
        "2x2 tensor torus."
    )

## 8. Next structural step

The finite-cluster projection is a controlled first stage.  The inferred parity rule is already periodic under the \(3\times2\) tile, and assigns a \(\mathbb Z_2\) charge to each of the 71 compressed physical states.  The next upgrade is to impose that charge directly on the tensor entries and virtual legs, eliminating the global projector.  A successful numerical tensor should then be converted into local equations for

\[
B\,\psi_+(A)=0,
\qquad
(V_+-v_0)\psi_+(A)=0,
\]

or an equivalent PEPS telescoping identity valid for arbitrary two-dimensional tori.